# Vehicle detector training

Fine-tunes YOLOv8 on the Roboflow *Vehicle-Detection v13* dataset (bus, car, motorcycle,
person, rickshaw, truck) and produces **README-ready** numbers.

## What changed vs. v1 (and why)

| Problem found in the original (v1) notebook | Fix here |
|---|---|
| 156 of 972 train images (and 3 val) were **silently ignored** by Ultralytics ("labels mix segment and detection rows") | Polygon rows are converted to tight boxes; every image is used |
| Same source images appeared in train/val/test (11 / 2 / 1, measured in v1) and video frames were split randomly, so near-identical frames leak across splits | Keep Roboflow's original (unaugmented) val/test, **purge every train scene group that overlaps them**, then *assert* zero source/scene overlap. A `regroup` mode re-balances rare classes instead |
| One model, one size, one resolution; augmentation turned off on an untested assumption | Small **experiment table** (n vs s, 640 vs 960, light vs default augmentation) selected on **val only** |
| Recall measured at conf 0.25 while the pipeline runs at 0.40 | Final metrics at the **pipeline operating point** (`OP_CONF`, `OP_NMS_IOU`) |
| README table not reproducible from the notebook | One cell emits the README table, with **bootstrap 95% CIs** and per-class test counts |
| Truck weakness blamed on "47 training instances" (47 is the *test* count) | Confusion analysis: what are ground-truth trucks predicted as? |

## How to run
Kaggle, GPU T4, internet on, secret `ROBOFLOW_API_KEY` attached. Set `QUICK = True` first
(~5 min) to smoke-test the whole notebook, then run it for real. Training is resumable:
finished experiments are skipped if their `best.pt` exists.

## Honest limitations
- The grouping rule is a **heuristic from filenames** (the export has no video IDs). In `purge` mode the
  notebook picks the coarsest grouping that keeps enough training data; the sweep table is printed in step 3.
  Finer groups keep more training data but let nearby frames of one scene sit on both sides: say which
  setting you used when you quote numbers.
- Val/test are small (~70 images each after de-duplication). Differences under ~0.01 mAP are noise;
  rare classes (truck) have wide intervals. Report them as such.
- One seed per experiment. For a claim like "960 beats 640", repeat the two finalists with 3 seeds.

In [ ]:
!pip install -q "ultralytics==8.4.138" roboflow

In [ ]:
import hashlib
import itertools
import json
import os
import random
import re
import shutil
import time
import warnings
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import yaml

QUICK = False  # True = one 3-epoch run to smoke-test the notebook end to end

SEED = 42
WORK = Path(os.environ.get("WORK_DIR", "/kaggle/working"))
RAW = WORK / "Vehicle-Detection-13"      # Roboflow export, untouched
CLEAN = WORK / "vehicle_clean"           # fixed labels + grouped split (this is what we train on)
RUNS_DIR = WORK / "runs"
ART = WORK / "artifacts"
DATA_YAML = CLEAN / "data.yaml"

RATIOS = {"train": 0.70, "val": 0.15, "test": 0.15}
IMG_EXTS = {".jpg", ".jpeg", ".png"}
MIN_BOX = 0.002                          # drop boxes thinner than 0.2% of the image

# "purge"   : keep Roboflow's original val/test (never augmented) and drop train scene groups that overlap them.
# "regroup" : pool everything and re-split by scene group, balancing rare classes. Caveat: Roboflow augments
#             only train, so most pooled images are augmented copies and val/test would contain some.
SPLIT_MODE = "purge"

# In purge mode the grouping coarseness is chosen from the data: the coarsest setting that still keeps at
# least MIN_RETAIN of the train images AND of every class's train instances. Set AUTO_BUCKETS = False to use
# GROUP_BUCKETS / DEFAULT_BUCKET as written.
AUTO_BUCKETS = True
MIN_RETAIN = 0.60

# Frames closer than this many indices (per filename prefix) are treated as one scene.
GROUP_BUCKETS = {"frame": 1000, "image": 1000}
DEFAULT_BUCKET = 50

# Evaluation operating point. Keep identical to configs/config.yaml so the README
# numbers describe the system you actually run.
OP_CONF = 0.40
OP_NMS_IOU = 0.50
MATCH_IOU = 0.50

EXPERIMENTS = [
    dict(name="n640_light",   weights="yolov8n.pt", imgsz=640, aug="light"),    # v1 recipe on the fixed split
    dict(name="n640_default", weights="yolov8n.pt", imgsz=640, aug="default"),
    dict(name="s640_default", weights="yolov8s.pt", imgsz=640, aug="default"),
    dict(name="s960_default", weights="yolov8s.pt", imgsz=960, aug="default"),  # if CUDA OOM: add batch=8
]
EPOCHS, PATIENCE = 100, 25
TIE_MARGIN = 0.01                        # val mAP50-95 differences below this are noise
if QUICK:
    EXPERIMENTS, EPOCHS, PATIENCE = EXPERIMENTS[:1], 3, 3

random.seed(SEED)
np.random.seed(SEED)

In [ ]:
if not RAW.exists():
    from kaggle_secrets import UserSecretsClient
    from roboflow import Roboflow

    rf = Roboflow(api_key=UserSecretsClient().get_secret("ROBOFLOW_API_KEY"))
    project = rf.workspace("ziauddin-university").project("vehicle-detection-y7ocl")
    project.version(13).download("yolov8", location=str(RAW))
else:
    print("Dataset already present:", RAW)

## 1. Audit the raw export
Counts what v1 only discovered through training warnings: label rows that are polygons, files that
mix polygons and boxes, and per-class totals.

In [ ]:
data_cfg = yaml.safe_load((RAW / "data.yaml").read_text())
NAMES = data_cfg["names"]
if isinstance(NAMES, dict):
    NAMES = [NAMES[k] for k in sorted(NAMES)]
NC = len(NAMES)
print("Classes:", NAMES)


def classify_row(parts):
    """'box' = cls cx cy w h ; 'polygon' = cls x1 y1 x2 y2 x3 y3 ... ; else 'invalid'."""
    n = len(parts) - 1
    if n == 4:
        return "box"
    if n >= 6 and n % 2 == 0:
        return "polygon"
    return "invalid"


def audit_split(split):
    img_dir, lbl_dir = RAW / split / "images", RAW / split / "labels"
    imgs = [p for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXTS]
    row_types, per_class = Counter(), Counter()
    mixed = missing = 0
    for p in imgs:
        lp = lbl_dir / (p.stem + ".txt")
        if not lp.exists():
            missing += 1
            continue
        types = set()
        for line in lp.read_text().splitlines():
            parts = line.split()
            if not parts:
                continue
            kind = classify_row(parts)
            row_types[kind] += 1
            types.add(kind)
            try:
                per_class[NAMES[int(float(parts[0]))]] += 1
            except (ValueError, IndexError):
                row_types["bad_class_id"] += 1
        mixed += len(types) > 1
    print(f"{split:6s} images={len(imgs):5d}  no_label_file={missing}  "
          f"files_mixing_box+polygon={mixed}  rows={dict(row_types)}")
    return per_class


for split in ["train", "valid", "test"]:
    pc = audit_split(split)
    print("       classes:", dict(sorted(pc.items())))

from PIL import Image
sizes = Counter()
for split in ["train", "valid", "test"]:
    for p in sorted((RAW / split / "images").iterdir())[:200]:
        if p.suffix.lower() in IMG_EXTS:
            with Image.open(p) as im:
                sizes[im.size] += 1
print("Image sizes (first 200 per split):", sizes.most_common(6))
print("Mixed resolutions: this is what imgsz=960 vs 640 is testing in the experiments below.")

## 2. Fix labels and pool the images
Polygon rows become their tight bounding box (that is exactly what a detector learns from a
segmentation label anyway). Out-of-range coordinates are clipped; degenerate boxes are dropped and
counted. Roboflow's export carries `.rf.<hash>` suffixes — images sharing the part before it are
augmented copies of the **same source image** (`base`), which is what lets us detect leakage.

In [ ]:
_RF = re.compile(r"\.rf\.[0-9a-f]+$")
_EXT_TAG = re.compile(r"_(jpg|jpeg|png)$", re.I)


def base_name(stem):
    """'frame11250_jpg.rf.64e7...' -> 'frame11250'  (the source image identity)."""
    return _EXT_TAG.sub("", _RF.sub("", stem))


def group_key(base, buckets=None, default=None):
    """Scene-level group: filename prefix + index bucket.

    Neighbouring video frames (frame11250, frame11251, ...) land in the same group so they cannot be
    split across train and test. Heuristic: the export carries no video IDs.
    """
    m = re.match(r"^(.*?)(\d+)\D*$", base)
    if not m:
        return f"misc:{base}"
    prefix = re.sub(r"[^A-Za-z]", "", m.group(1)).lower() or "num"
    buckets = GROUP_BUCKETS if buckets is None else buckets
    default = DEFAULT_BUCKET if default is None else default
    return f"{prefix}:{int(m.group(2)) // buckets.get(prefix, default)}"


def clean_label_lines(lines, nc):
    out, stats = [], Counter()
    for line in lines:
        parts = line.split()
        if not parts:
            continue
        kind = classify_row(parts)
        try:
            cid = int(float(parts[0]))
            vals = [float(v) for v in parts[1:]]
        except ValueError:
            stats["invalid"] += 1
            continue
        if kind == "invalid" or not 0 <= cid < nc:
            stats["invalid"] += 1
            continue
        if kind == "polygon":
            xs, ys = vals[0::2], vals[1::2]
            x1, x2, y1, y2 = min(xs), max(xs), min(ys), max(ys)
            stats["polygon_to_box"] += 1
        else:
            cx, cy, w, h = vals
            x1, x2, y1, y2 = cx - w / 2, cx + w / 2, cy - h / 2, cy + h / 2
        x1, y1, x2, y2 = (min(max(v, 0.0), 1.0) for v in (x1, y1, x2, y2))
        w, h = x2 - x1, y2 - y1
        if w < MIN_BOX or h < MIN_BOX:
            stats["degenerate_dropped"] += 1
            continue
        out.append(f"{cid} {(x1 + x2) / 2:.6f} {(y1 + y2) / 2:.6f} {w:.6f} {h:.6f}")
        stats["kept"] += 1
    return out, stats


ITEMS, tot = [], Counter()
for split in ["train", "valid", "test"]:
    for p in sorted((RAW / split / "images").iterdir()):
        if p.suffix.lower() not in IMG_EXTS:
            continue
        lp = RAW / split / "labels" / (p.stem + ".txt")
        if not lp.exists():
            tot["no_label_file"] += 1
            continue
        lines, st = clean_label_lines(lp.read_text().splitlines(), NC)
        tot.update(st)
        if not lines:
            tot["no_boxes_after_clean"] += 1
            continue
        base = base_name(p.stem)
        ITEMS.append(dict(path=p, base=base, group=group_key(base), lines=lines, origin=split))

print("Label rows:", dict(tot))
print(f"Usable image files: {len(ITEMS)}  |  unique source images: {len({i['base'] for i in ITEMS})}")
assert len({i["path"].name for i in ITEMS}) == len(ITEMS), "filename collision across raw splits"

## 3. Leak-free split
Images → source images (`base`) → scene groups (`group`; neighbouring video frames share one).

**`purge` (default)** — Roboflow's original `valid` and `test` images are never augmented, so they are the
honest evaluation set. Any train image whose scene group also appears in val/test is dropped, and val loses
groups that appear in test. You lose some training data; the printout says how much.

**`regroup`** — pools everything, assigns whole groups to train/val/test over many random tries, and keeps the
assignment whose per-class shares are closest to 70/15/15 (helps rare classes such as truck). Val/test keep one
image per source image, preferring an unaugmented one, but most pooled images exist only as Roboflow-augmented
train copies, so some val/test images carry blur/noise/grayscale. Use it only if `purge` leaves a class too thin.

In [ ]:
BASES = defaultdict(list)
for it in ITEMS:
    BASES[it["base"]].append(it)

base_counts = {}
for b, files in BASES.items():
    counts = np.zeros(NC)
    for line in files[0]["lines"]:
        counts[int(line.split()[0])] += 1
    base_counts[b] = counts


def grouping_stats(frame_bucket, default_bucket):
    """How much training data survives the purge if scenes are grouped this coarsely?"""
    buckets = {"frame": frame_bucket, "image": frame_bucket}
    grp = {b: group_key(b, buckets, default_bucket) for b in BASES}
    origins = {b: {it["origin"] for it in files} for b, files in BASES.items()}
    eval_groups = {grp[b] for b in BASES if origins[b] & {"valid", "test"}}
    pool = [b for b in BASES if "train" in origins[b]]
    kept = [b for b in pool if grp[b] not in eval_groups]
    total = np.maximum(sum(base_counts[b] for b in pool), 1)
    kept_cls = sum((base_counts[b] for b in kept), np.zeros(NC))
    return dict(frame=frame_bucket, default=default_bucket, groups=len(set(grp.values())),
                img_kept=len(kept) / len(pool), cls_kept=kept_cls / total)


if SPLIT_MODE == "purge" and AUTO_BUCKETS:
    CANDIDATES = [(1000, 50), (500, 25), (250, 25), (250, 10), (100, 10), (50, 5), (20, 2)]  # coarse -> fine
    sweep = [grouping_stats(f, d) for f, d in CANDIDATES]
    print(f"{'frame':>6s} {'other':>6s} {'groups':>7s} {'train kept':>11s} | train instances kept per class")
    for r in sweep:
        print(f"{r['frame']:6d} {r['default']:6d} {r['groups']:7d} {r['img_kept']:11.0%} | "
              + "  ".join(f"{n[:5]}={v:.0%}" for n, v in zip(NAMES, r["cls_kept"])))
    ok = [r for r in sweep if r["img_kept"] >= MIN_RETAIN and r["cls_kept"].min() >= MIN_RETAIN]
    chosen = ok[0] if ok else sweep[-1]
    if not ok:
        print(f"WARNING: no setting keeps >= {MIN_RETAIN:.0%} of every class; using the finest. "
              "Some classes are concentrated in scenes that also appear in test.")
    GROUP_BUCKETS = {"frame": chosen["frame"], "image": chosen["frame"]}
    DEFAULT_BUCKET = chosen["default"]
    print(f"Chosen grouping: frame/image bucket={chosen['frame']}, other bucket={chosen['default']}")

for it in ITEMS:                      # (re)compute groups with the final buckets
    it["group"] = group_key(it["base"])
base_info = {b: dict(group=files[0]["group"], counts=base_counts[b]) for b, files in BASES.items()}


def stratified_group_split(base_info, ratios, seed, tries=500):
    groups = defaultdict(list)
    for b, info in base_info.items():
        groups[info["group"]].append(b)
    keys = sorted(groups)
    total_imgs = len(base_info)
    total_cls = np.maximum(sum(i["counts"] for i in base_info.values()), 1)
    rng = random.Random(seed)
    best = (float("inf"), None)

    for _ in range(tries):
        order = sorted(keys, key=lambda k: -(len(groups[k]) + rng.random()))
        n_img = {s: 0 for s in ratios}
        cls = {s: np.zeros(NC) for s in ratios}
        assign = {s: [] for s in ratios}
        for k in order:
            s = max(ratios, key=lambda s: ratios[s] * total_imgs - n_img[s])
            assign[s].extend(groups[k])
            n_img[s] += len(groups[k])
            for b in groups[k]:
                cls[s] += base_info[b]["counts"]
        if any(n_img[s] == 0 for s in ratios):
            continue
        score = sum(abs(n_img[s] / total_imgs - r) for s, r in ratios.items())
        score += sum(np.abs(cls[s] / total_cls - r).mean() for s, r in ratios.items())
        if score < best[0]:
            best = (score, assign)
    assert best[1] is not None, "could not build a non-empty split; too few groups - lower GROUP_BUCKETS"
    return best[1], best[0]


def purge_split(base_info):
    ev = {"val": set(), "test": set()}
    for b, files in BASES.items():
        origins = {it["origin"] for it in files}
        if "test" in origins:
            ev["test"].add(b)
        elif "valid" in origins:
            ev["val"].add(b)
    test_groups = {base_info[b]["group"] for b in ev["test"]}
    dropped_val = {b for b in ev["val"] if base_info[b]["group"] in test_groups}
    ev["val"] -= dropped_val
    eval_groups = test_groups | {base_info[b]["group"] for b in ev["val"]}
    train = [b for b in base_info if base_info[b]["group"] not in eval_groups]
    purged = len(base_info) - len(train) - len(ev["val"]) - len(ev["test"]) - len(dropped_val)
    print(f"purge: kept {len(train)} train source images; removed {purged} train + {len(dropped_val)} val "
          f"source images that shared a scene group with val/test")
    if purged > 0.25 * (len(train) + purged):
        print("WARNING: >25% of train was purged. Groups are probably too coarse: lower GROUP_BUCKETS / DEFAULT_BUCKET.")
    return {"train": train, "val": sorted(ev["val"]), "test": sorted(ev["test"])}


if SPLIT_MODE == "purge":
    ASSIGN = purge_split(base_info)
else:
    ASSIGN, score = stratified_group_split(base_info, RATIOS, SEED)
    print(f"Chosen split score (lower is better): {score:.3f}")

gsz = Counter(info["group"] for info in base_info.values())
print(f"{len(gsz)} scene groups; largest 8:", gsz.most_common(8))
print("Inspect this: if one group holds most images, shrink its bucket in GROUP_BUCKETS.")


def materialize(assign, seed):
    rng = random.Random(seed)
    if CLEAN.exists():
        shutil.rmtree(CLEAN)
    manifest, aug_in_eval = {}, 0
    for split, bases in assign.items():
        (CLEAN / split / "images").mkdir(parents=True)
        (CLEAN / split / "labels").mkdir(parents=True)
        files = []
        for b in sorted(bases):
            cands = sorted(BASES[b], key=lambda it: it["path"].name)
            if split == "train":
                files += cands
            else:
                exact = [it for it in cands if it["origin"] == {"val": "valid", "test": "test"}[split]]
                originals = [it for it in cands if it["origin"] != "train"]
                aug_in_eval += not (exact or originals)
                files.append(rng.choice(exact or originals or cands))
        for it in files:
            dst = CLEAN / split / "images" / it["path"].name
            try:
                os.link(it["path"], dst)
            except OSError:
                shutil.copy2(it["path"], dst)
            (CLEAN / split / "labels" / (it["path"].stem + ".txt")).write_text("\n".join(it["lines"]) + "\n")
        manifest[split] = sorted(it["path"].name for it in files)
    print(f"val/test images that are only available as an augmented copy: {aug_in_eval}")
    return manifest


MANIFEST = materialize(ASSIGN, SEED)
SPLIT_TAG = hashlib.md5(json.dumps(MANIFEST, sort_keys=True).encode()).hexdigest()[:6]
print("Split fingerprint:", SPLIT_TAG, "(run folders are tagged with it, so weights from another split are never reused)")

yaml.safe_dump(
    {"path": str(CLEAN), "train": "train/images", "val": "val/images", "test": "test/images",
     "nc": NC, "names": NAMES},
    DATA_YAML.open("w"), sort_keys=False)

In [ ]:
# Leakage checks: these must pass, otherwise the evaluation is not trustworthy.
def files_of(split):
    return [p for p in (CLEAN / split / "images").iterdir() if p.suffix.lower() in IMG_EXTS]


bases_by = {s: {base_name(p.stem) for p in files_of(s)} for s in RATIOS}
groups_by = {s: {group_key(b) for b in bases_by[s]} for s in RATIOS}
for a, b in itertools.combinations(RATIOS, 2):
    assert not (bases_by[a] & bases_by[b]), f"source-image overlap between {a} and {b}"
    assert not (groups_by[a] & groups_by[b]), f"scene-group overlap between {a} and {b}"
for s in ("val", "test"):
    assert len(bases_by[s]) == len(files_of(s)), f"duplicate source images inside {s}"
print("OK: no source-image or scene-group overlap between splits; val/test are de-duplicated.\n")

print(f"{'split':6s} {'images':>7s} " + " ".join(f"{n[:10]:>10s}" for n in NAMES))
for s in RATIOS:
    per = Counter()
    for lp in (CLEAN / s / "labels").glob("*.txt"):
        for line in lp.read_text().splitlines():
            per[int(line.split()[0])] += 1
    print(f"{s:6s} {len(files_of(s)):7d} " + " ".join(f"{per[i]:10d}" for i in range(NC)))
    if s != "train":
        low = [NAMES[i] for i in range(NC) if per[i] < 100]
        if low:
            print(f"       ^ WARNING: <100 instances in {s} for {low}: treat those metrics as low-confidence")

## 4. Evaluation helpers
Ultralytics' `val()` gives mAP, but the system runs at a fixed confidence threshold, so we also
measure **precision and recall at that operating point** with one-to-one greedy matching, plus a
**bootstrap over images** for confidence intervals. These helpers are plain NumPy (unit-testable).

In [ ]:
def iou_matrix(a, b):
    """a: (N,4) xyxy, b: (M,4) xyxy -> (N,M) IoU."""
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0])
    y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2])
    y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)


def evaluate_image(gt_cls, gt_box, pr_cls, pr_conf, pr_box, n_cls, iou_thr):
    """Class-aware, one-to-one greedy matching (highest confidence first).

    Returns tp, fp, fn per class and a GT-class x (pred-class | missed) confusion matrix, where each GT
    is assigned the class of the best-overlapping prediction of ANY class (IoU >= iou_thr).
    """
    tp, fp, fn = np.zeros(n_cls), np.zeros(n_cls), np.zeros(n_cls)
    conf = np.zeros((n_cls, n_cls + 1))
    ious = iou_matrix(pr_box, gt_box)
    matched = set()
    for p in np.argsort(-pr_conf):
        best_g, best_iou = None, iou_thr
        for g in range(len(gt_cls)):
            if g in matched or gt_cls[g] != pr_cls[p] or ious[p, g] < best_iou:
                continue
            best_g, best_iou = g, ious[p, g]
        if best_g is None:
            fp[pr_cls[p]] += 1
        else:
            matched.add(best_g)
            tp[pr_cls[p]] += 1
    for g in range(len(gt_cls)):
        if g not in matched:
            fn[gt_cls[g]] += 1
        if len(pr_cls) and ious[:, g].max() >= iou_thr:
            conf[gt_cls[g], pr_cls[int(ious[:, g].argmax())]] += 1
        else:
            conf[gt_cls[g], n_cls] += 1
    return tp, fp, fn, conf


def evaluate_split(gts, preds, n_cls, iou_thr, only_classes=None):
    rows = []
    for (g_cls, g_box), (p_cls, p_conf, p_box) in zip(gts, preds):
        if only_classes is not None:
            keep = np.isin(g_cls, list(only_classes))
            g_cls, g_box = g_cls[keep], g_box[keep]
        rows.append(evaluate_image(g_cls, g_box, p_cls, p_conf, p_box, n_cls, iou_thr))
    tp, fp, fn = (np.stack([r[i] for r in rows]) for i in range(3))
    return tp, fp, fn, sum(r[3] for r in rows)


def _ratio(num, den):
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(den > 0, num / den, np.nan)


def point_and_ci(tp, fp, fn, n_boot=1000, seed=0):
    """Per-class recall/precision (pooled over images) with 95% bootstrap-over-images intervals."""
    rec = _ratio(tp.sum(0), (tp + fn).sum(0))
    prec = _ratio(tp.sum(0), (tp + fp).sum(0))
    idx = np.random.default_rng(seed).integers(0, len(tp), size=(n_boot, len(tp)))
    TP, FP, FN = tp[idx].sum(1), fp[idx].sum(1), fn[idx].sum(1)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        r_lo, r_hi = np.nanpercentile(_ratio(TP, TP + FN), [2.5, 97.5], axis=0)
        p_lo, p_hi = np.nanpercentile(_ratio(TP, TP + FP), [2.5, 97.5], axis=0)
    return dict(recall=rec, recall_ci=(r_lo, r_hi), precision=prec, precision_ci=(p_lo, p_hi))


def load_gt(split):
    """Ground truth of a split as [(cls[N], xyxy_pixels[N,4])] aligned with sorted image paths."""
    from PIL import Image
    paths = sorted(p for p in (CLEAN / split / "images").iterdir() if p.suffix.lower() in IMG_EXTS)
    gts = []
    for p in paths:
        w, h = Image.open(p).size
        arr = np.loadtxt(CLEAN / split / "labels" / (p.stem + ".txt"), ndmin=2)
        if arr.size == 0:
            gts.append((np.zeros(0, int), np.zeros((0, 4))))
            continue
        cx, cy, bw, bh = arr[:, 1] * w, arr[:, 2] * h, arr[:, 3] * w, arr[:, 4] * h
        gts.append((arr[:, 0].astype(int),
                    np.stack([cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2], 1)))
    return paths, gts


def predict_split(model, paths, imgsz, conf, nms_iou, device, class_map=None, batch=16):
    """Runs the model; optionally remaps/filters class ids (used for the COCO baseline)."""
    preds = []
    for i in range(0, len(paths), batch):
        results = model.predict([str(p) for p in paths[i:i + batch]], imgsz=imgsz, conf=conf,
                                iou=nms_iou, device=device, verbose=False)
        for r in results:
            b = r.boxes
            cls = b.cls.cpu().numpy().astype(int)
            cf, xy = b.conf.cpu().numpy(), b.xyxy.cpu().numpy()
            if class_map is not None:
                keep = np.array([c in class_map for c in cls], dtype=bool)
                cls, cf, xy = np.array([class_map[c] for c in cls[keep]], dtype=int), cf[keep], xy[keep]
            preds.append((cls, cf, xy))
    return preds

## 5. Experiments
Same data, same seed, same epochs; only the model size, resolution and augmentation change.
`light` = the v1 recipe (HSV/flip off because Roboflow already augments train). `default` = Ultralytics
defaults — an **ablation**, not an assumption. Selection uses the **val** split only; test is touched once, later.

In [ ]:
import pandas as pd
import torch
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == 0 else "")

AUG = {
    "light": dict(hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, fliplr=0.0, mosaic=0.5, mixup=0.0),
    "default": dict(),
}
TRUCK = next(i for i, n in enumerate(NAMES) if n.lower() == "truck")
BEST = {}


def run_experiment(exp):
    name = f"{exp['name']}_{SPLIT_TAG}"
    expected = RUNS_DIR / name / "weights" / "best.pt"
    last = RUNS_DIR / name / "weights" / "last.pt"
    minutes = float("nan")
    if expected.exists():
        print(f"[skip] {name}: found {expected}")
        BEST[name] = expected
    else:
        t0 = time.time()
        if last.exists():                      # Kaggle session died mid-run: continue it
            print(f"[resume] {name} from {last}")
            model = YOLO(str(last))
            model.train(resume=True)
        else:
            model = YOLO(exp["weights"])
            model.train(data=str(DATA_YAML), epochs=EPOCHS, patience=PATIENCE, imgsz=exp["imgsz"],
                        batch=exp.get("batch", 16), device=DEVICE, project=str(RUNS_DIR), name=name,
                        exist_ok=True, seed=SEED, deterministic=True, workers=2, save_period=5,
                        **AUG[exp["aug"]])
        minutes = (time.time() - t0) / 60
        BEST[name] = Path(model.trainer.save_dir) / "weights" / "best.pt"

    m = YOLO(str(BEST[name])).val(data=str(DATA_YAML), split="val", imgsz=exp["imgsz"],
                                  device=DEVICE, plots=False, verbose=False)
    return dict(name=name, model=exp["weights"], imgsz=exp["imgsz"], aug=exp["aug"],
                val_map50=float(m.box.map50), val_map50_95=float(m.box.map),
                truck_ap50_95=float(m.box.maps[TRUCK]),
                ms_per_img=float(m.speed["inference"]), train_min=minutes)


ROWS = [run_experiment(e) for e in EXPERIMENTS]
RESULTS = pd.DataFrame(ROWS).round(4)
RESULTS.to_csv(WORK / "experiments.csv", index=False)
RESULTS

In [ ]:
top = RESULTS.val_map50_95.max()
tied = RESULTS[RESULTS.val_map50_95 >= top - TIE_MARGIN]
WINNER = tied.sort_values("ms_per_img").iloc[0]
print(f"Best val mAP50-95: {top:.4f}. Candidates within {TIE_MARGIN} of it:", list(tied.name))
print(f"Winner (fastest among near-ties): {WINNER['name']}  imgsz={WINNER['imgsz']}  "
      f"val mAP50-95={WINNER['val_map50_95']:.4f}  {WINNER['ms_per_img']:.1f} ms/img")

## 6. Final evaluation on the held-out test split (run once)
- Ultralytics test mAP (threshold-free).
- Precision / recall **at the pipeline operating point** with bootstrap 95% CIs.
- The same measurement for the pretrained COCO model on the classes COCO has (rickshaw is excluded:
  COCO has no such class, so it cannot be scored as a miss).
- Where do ground-truth trucks go? (confusion at IoU ≥ 0.5)

In [ ]:
imgsz = int(WINNER["imgsz"])
best_pt = BEST[WINNER["name"]]
model = YOLO(str(best_pt))

official = model.val(data=str(DATA_YAML), split="test", imgsz=imgsz, device=DEVICE, plots=True, verbose=False)
print(f"Test mAP50={official.box.map50:.4f}  mAP50-95={official.box.map:.4f}")

from IPython.display import FileLink, Image as IPImage, display
for png in (best_pt.parent.parent / "results.png", Path(official.save_dir) / "confusion_matrix_normalized.png"):
    if png.exists():
        display(IPImage(filename=str(png)))

paths, gts = load_gt("test")
n_gt = np.zeros(NC, int)
for c, _ in gts:
    n_gt += np.bincount(c, minlength=NC)

ft_preds = predict_split(model, paths, imgsz, OP_CONF, OP_NMS_IOU, DEVICE)
ft_tp, ft_fp, ft_fn, ft_conf = evaluate_split(gts, ft_preds, NC, MATCH_IOU)
ft = point_and_ci(ft_tp, ft_fp, ft_fn)

# Pretrained COCO baseline at the same operating point, restricted to classes COCO has.
idx = {re.sub(r"[ _]", "", n.lower()): i for i, n in enumerate(NAMES)}
COCO_TO_CUSTOM = {0: idx["person"], 2: idx["car"], 3: idx["motorcycle"], 5: idx["bus"], 7: idx["truck"]}
coco = YOLO("yolov8n.pt")
pt_preds = predict_split(coco, paths, 640, OP_CONF, OP_NMS_IOU, DEVICE, class_map=COCO_TO_CUSTOM)
pt_tp, pt_fp, pt_fn, _ = evaluate_split(gts, pt_preds, NC, MATCH_IOU, only_classes=set(COCO_TO_CUSTOM.values()))
pt = point_and_ci(pt_tp, pt_fp, pt_fn)


def fmt(v, lo, hi):
    return "n/a" if np.isnan(v) else f"{v:.2f} [{lo:.2f}-{hi:.2f}]"


ap50 = {i: float(official.box.all_ap[list(official.box.ap_class_index).index(i), 0])
        if i in official.box.ap_class_index else float("nan") for i in range(NC)}
ap = {i: float(official.box.maps[i]) for i in range(NC)}

lines = [f"Test split, {len(paths)} images; operating point conf={OP_CONF}, NMS IoU={OP_NMS_IOU}, "
         f"match IoU={MATCH_IOU}. Brackets = 95% bootstrap CI over images.", "",
         "| Class | GT boxes | Pretrained COCO recall | Fine-tuned recall | Fine-tuned precision | AP50 | AP50-95 |",
         "|---|---|---|---|---|---|---|"]
for i, n in enumerate(NAMES):
    pre = "N/A (no COCO class)" if i not in COCO_TO_CUSTOM.values() else fmt(
        pt["recall"][i], pt["recall_ci"][0][i], pt["recall_ci"][1][i])
    lines.append(f"| {n} | {n_gt[i]} | {pre} | "
                 f"{fmt(ft['recall'][i], ft['recall_ci'][0][i], ft['recall_ci'][1][i])} | "
                 f"{fmt(ft['precision'][i], ft['precision_ci'][0][i], ft['precision_ci'][1][i])} | "
                 f"{ap50[i]:.2f} | {ap[i]:.2f} |")
lines += ["", f"Overall test mAP50 = {official.box.map50:.3f}, mAP50-95 = {official.box.map:.3f} "
              f"(model: {WINNER['name']}, imgsz {imgsz}, {WINNER['ms_per_img']:.1f} ms/img on {DEVICE})."]
TABLE_MD = "\n".join(lines)
print(TABLE_MD)

print("\nWhere do ground-truth trucks go (IoU >= 0.5, best-overlapping prediction of any class)?")
row = ft_conf[TRUCK]
for j, label in enumerate(NAMES + ["missed (no overlapping box)"]):
    if row[j]:
        print(f"  {label:32s} {row[j] / row.sum():5.1%}  ({int(row[j])})")

## 7. Save artifacts
Everything needed to reproduce and publish: weights, the README table, the split manifest, versions.
Upload `best.pt` to a GitHub Release (it is gitignored) and commit the rest of `artifacts/` except the zip.

In [ ]:
import platform

import ultralytics
from IPython.display import FileLink, display

ART.mkdir(exist_ok=True)
shutil.copy(best_pt, ART / "best.pt")
shutil.copy(DATA_YAML, ART / "data.yaml")
shutil.copy(WORK / "experiments.csv", ART / "experiments.csv")
run_dir = best_pt.parent.parent
# Training curves, labels.jpg, args.yaml, P/R/F1 curves, val_batch samples: what reports/ in the repo is built from.
shutil.copytree(run_dir, ART / "winner_run", ignore=shutil.ignore_patterns("weights", "epoch*.pt"), dirs_exist_ok=True)
shutil.copytree(official.save_dir, ART / "test_eval", dirs_exist_ok=True)


def _clean(x):
    if isinstance(x, dict):
        return {k: _clean(v) for k, v in x.items()}
    if isinstance(x, (tuple, list)):
        return [_clean(v) for v in x]
    arr = np.asarray(x)
    return [None if np.isnan(v) else float(v) for v in arr.ravel()] if arr.ndim else (None if np.isnan(arr) else float(arr))


(ART / "comparison.json").write_text(json.dumps(
    {"classes": NAMES, "n_gt": n_gt.tolist(), "pretrained_coco": _clean(pt), "finetuned": _clean(ft)}, indent=1))
(ART / "results_table.md").write_text(TABLE_MD + "\n")
(ART / "split_manifest.json").write_text(json.dumps(MANIFEST, indent=1))
(ART / "run_meta.json").write_text(json.dumps({
    "winner": WINNER.to_dict(), "seed": SEED, "epochs": EPOCHS, "patience": PATIENCE,
    "dataset": "roboflow ziauddin-university/vehicle-detection-y7ocl v13",
    "ratios": RATIOS, "group_buckets": GROUP_BUCKETS, "default_bucket": DEFAULT_BUCKET,
    "operating_point": dict(conf=OP_CONF, nms_iou=OP_NMS_IOU, match_iou=MATCH_IOU),
    "test": dict(map50=float(official.box.map50), map50_95=float(official.box.map)),
    "versions": dict(python=platform.python_version(), torch=torch.__version__,
                     ultralytics=ultralytics.__version__),
}, indent=2, default=str))

shutil.make_archive(str(WORK / "training_artifacts"), "zip", ART)
shutil.make_archive(str(WORK / "winner_run_full"), "zip", run_dir)   # incl. weights/, like v1's phase3_artifacts.zip
print("Saved:", sorted(p.name for p in ART.iterdir()))
display(FileLink("training_artifacts.zip"))
display(FileLink("winner_run_full.zip"))